In [1]:
# Install required packages (run in Colab or your environment)
!pip install -qqq "transformers>=4.55.0" "trl>=0.22.1" "datasets" "torch"
!pip install -qqq "accelerate" "peft" "trackio" "huggingface_hub"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 103.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 156.3 MB/s eta 0:00:00a 0:00:01


In [2]:
import transformers, trl, datasets, torch
print(transformers.__version__, trl.__version__, datasets.__version__, torch.__version__)

4.57.6 0.27.1 4.0.0 2.9.0+cu126


In [3]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")

# Authenticate with Hugging Face (optional, for private models)
from huggingface_hub import login
login()  # Uncomment if you need to access private models


Using CUDA GPU: NVIDIA A100-SXM4-80GB
GPU memory: 85.2GB


### Load SmolLM3 Models

Now let's load the base and instruct models for comparison.


In [4]:
# Load both base and instruct models for comparison
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

# Load tokenizers
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Load models (use smaller precision for memory efficiency)
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)

# Load instruct model
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)

print("Models loaded successfully!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Models loaded successfully!


### Compare Base vs Instruct Model Responses


In [5]:
# Test the same prompt on both models
test_prompt = "Explain quantum computing in simple terms."

# Prepare the prompt for base model (no chat template)
base_inputs = base_tokenizer(test_prompt, return_tensors="pt").to(device)

# Prepare the prompt for instruct model (with chat template)
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt}
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
instruct_inputs = instruct_tokenizer(instruct_formatted, return_tensors="pt").to(device)

# Generate responses
print("=== Model comparison ===\n")

print("🤖 BASE MODEL RESPONSE:")
with torch.no_grad():
    base_outputs = base_model.generate(
        **base_inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id,
    )
    base_response = base_tokenizer.decode(base_outputs[0], skip_special_tokens=True)
    print(base_response[len(test_prompt):])  # Show only the generated part

print("\n" + "=" * 50)
print("Instruct model response:")
with torch.no_grad():
    # # TODO Begin 
    # # Generate response from instruct model (20 points)
    # # instruct_outputs = instruct_model.generate(...)
    # # TODO End
    instruct_outputs = instruct_model.generate(
        **instruct_inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=instruct_tokenizer.eos_token_id,
    )
    instruct_response = instruct_tokenizer.decode(
        instruct_outputs[0], skip_special_tokens=False # Note: specical tokens such as <|im_start|>
    )
    # Extract only the assistant's response
    assistant_start = instruct_response.find("<|im_start|>assistant\n") + len(
        "<|im_start|>assistant\n"
    )
    assistant_response = instruct_response[assistant_start::].split("<|im_end|>")[0]
    print(assistant_response)


=== Model comparison ===

🤖 BASE MODEL RESPONSE:
 

1 Answer

giorgiana1976's profile pic

giorgiana1976 | College Teacher | (Level 3) Valedictorian

Posted on

Quantum computing is a new branch of computer science, which is based on quantum theory.

Quantum theory is the branch of physics, which studies the physical properties of physical systems, which are smaller than the atomic scale.

We know that the smallest physical system is the atom, which has a nucleus at it's center and electrons rotating around the nucleus. Quantum computing uses not the classical computers, but the quantum computers, which are based on the properties of the quantum physics.

Quantum computers are more powerful than classical computers because they are based on superposition principle and entanglement

Instruct model response:
<think>

</think>
Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations. In classical computing, a bit is a basic unit of inf

### Customizing Data into Chat-Format and the GSM8K Dataset Example

In [6]:
# Function to process different dataset formats
def process_qa_dataset(examples, question_col, answer_col):
    """Process Q&A datasets into chat format"""
    processed = []

    for question, answer in zip(examples[question_col], examples[answer_col]):
        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        processed.append(messages)

    return {"messages": processed}


def process_instruction_dataset(examples):
    """Process instruction-following datasets"""
    processed = []

    for instruction, response in zip(examples["instruction"], examples["response"]):
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response},
        ]
        processed.append(messages)

    return {"messages": processed}


# Example: Process GSM8K math dataset
print("=== PROCESSING GSM8K DATASET ===\n")

gsm8k = load_dataset(
    "openai/gsm8k", "main", split="train[:100]"
)  # Small subset for demo
print(f"Original GSM8K example: {gsm8k[0]}")

print("\n" + "=" * 50)

# Convert to chat format
def process_gsm8k(examples):
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        messages = [
            {
                "role": "system",
                "content": "You are a math tutor. Solve problems step by step.",
            },
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        processed.append(messages)
    return {"messages": processed}


gsm8k_processed = gsm8k.map(
    process_gsm8k, batched=True, remove_columns=gsm8k.column_names
)
print(f"Processed example: {gsm8k_processed[0]}")


=== PROCESSING GSM8K DATASET ===



README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Original GSM8K example: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}



Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Processed example: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Applying Chat Template to the Processed Chat-formatted Dataset

In [7]:
# Function to apply chat templates to processed datasets
def apply_chat_template_to_dataset(dataset, tokenizer):
    """Apply chat template to dataset for training"""

    def format_messages(examples):
        formatted_texts = []

        for messages in examples["messages"]:
            # # TODO Begin
            # # Apply chat template (20 points)
            # # HINT: Use tokenizer.apply_chat_template
            # # formatted_text = ...
            # # formatted_texts.append(formatted_text)
            # # TODO End
            formatted_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,  # We want the complete conversation for fine-tuning instead of inference
            )
            formatted_texts.append(formatted_text)
        return {"text": formatted_texts}

    return dataset.map(format_messages, batched=True)


# Apply to our processed GSM8K dataset
gsm8k_formatted = apply_chat_template_to_dataset(gsm8k_processed, instruct_tokenizer)
print("=== FORMATTED TRAINING DATA ===")
print(gsm8k_formatted[0]["text"])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

=== FORMATTED TRAINING DATA ===
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 01 February 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



### SmolTalk2 Dataset Inspection and Preparation


In [8]:
from datasets import load_dataset

# Load and prepare training dataset
print("=== PREPARING DATASET ===\n")

# Use SmolTalk2
dataset_dict = load_dataset("HuggingFaceTB/smoltalk2", "SFT")
print(f"Total splits: {len(dataset_dict)}")
print(f"Available splits: {list(dataset_dict.keys())}")
print(f"Number of total rows: {sum([dataset_dict[d].num_rows for d in dataset_dict])}")
print(f"Dataset structure: {dataset_dict}")

training_split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
print("NUmber of rows in this training split: ", len(dataset_dict[training_split]))

train_dataset = dataset_dict[training_split].select(range(2000))  # Use subset for faster training


=== PREPARING DATASET ===



README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

...

Generating table_gpt_no_think split:   0%|          | 0/13203 [00:00<?, ? examples/s]

Generating tulu_3_sft_personas_instruction_following_no_think split:   0%|          | 0/29970 [00:00<?, ? exam…

Generating xlam_traces_no_think split:   0%|          | 0/59962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/105 [00:00<?, ?it/s]

Total splits: 25
Available splits: ['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_t

In [9]:
print(f"Processed example: {train_dataset[0]}")

Processed example: {'messages': [{'content': 'Hi there', 'role': 'user'}, {'content': '<think>\nOkay, the user sent "Hi there". That\'s a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask how I can assist them. Let me make sure the tone is warm and approachable. Alright, something like "Hello! How can I assist you today?" That should work. Let me confirm there\'s no typo and it\'s in a natural, conversational style.\n</think>\n\nHello! How can I assist you today?', 'role': 'assistant'}, {'content': "I'm looking for a healthy breakfast idea. What's a good option?", 'role': 'user'}, {'content': "<think>\nOkay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats, and some carbs. Let me brainstorm some options.\n\nMaybe start with a classic like avocado toast.

In [10]:
# Prepare the dataset for SFT
def format_chat_template(example):
    """Format the messages using the chat template"""
    if "messages" in example:
        # SmolTalk2 format
        messages = example["messages"]
    else:
        # Custom format - adapt as needed
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]}
        ]
    
    # Apply chat template
    text = instruct_tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Apply formatting
formatted_dataset = train_dataset.map(format_chat_template, batched=True)
formatted_dataset = formatted_dataset.remove_columns(
    [col for col in formatted_dataset.column_names if col != "text"]
)
print(f"Formatted example: {formatted_dataset[0]['text']}...")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Formatted example: <|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 01 February 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, a

### LoRA SFT with TRL + SmolLM3



In [11]:
# Import required libraries for fine-tuning
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig
import torch

# Load SmolLM3 model for fine-tuning
raw_model_name = "HuggingFaceTB/SmolLM3-3B"
# raw_model_name = "HuggingFaceTB/SmolLM3-3B-base"
lora_instruct_model_name = "SmolLM3-Custom-SFT"

print(f"Loading {raw_model_name}...")
raw_model = AutoModelForCausalLM.from_pretrained(
    raw_model_name,
    dtype=torch.float16,  # Use float16 for memory efficiency
    device_map="auto",
    trust_remote_code=True,
)

raw_tokenizer = AutoTokenizer.from_pretrained(raw_model_name)

print(f"Model loaded! Parameters: {raw_model.num_parameters():,}")
print("Check special tokens:", raw_tokenizer.bos_token, raw_tokenizer.eos_token, raw_tokenizer.pad_token)

Loading HuggingFaceTB/SmolLM3-3B...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded! Parameters: 3,075,098,624
Check special tokens: None <|im_end|> <|im_end|>


In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

# Configure training parameters
training_config = SFTConfig(
    # Model and data
    output_dir=f"./{lora_instruct_model_name}",
    dataset_text_field="text",
    max_length=2048,
    # assistant_only_loss=True, # added (only train on the assistant generation)

    # Training hyperparameters
    per_device_train_batch_size=2,  # Adjust based on your GPU memory
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=1,  # Start with 1 epoch
    max_steps=1000,  # Limit steps for demo

    # Optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,

    # Memory optimization
    dataloader_num_workers=0,
    group_by_length=True,  # Group similar length sequences
    report_to = "none",

    # # Hugging Face Hub integration
    # push_to_hub=False,  # Set to True to upload to Hub
    # hub_model_id=f"your-username/{new_model_name}",

    # # Experiment tracking
    # report_to=["trackio"],  # Use trackio for experiment tracking
    # run_name=f"{new_model_name}-training",
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 4


In [13]:
# Create SFTTrainer with LoRA enabled

# lora_instruct_model = get_peft_model(raw_model, peft_config) # way-1
# lora_instruct_model.print_trainable_parameters() # way-1

lora_trainer = SFTTrainer(
    # model=lora_instruct_model, # way-1
    model = raw_model, # way-2
    # train_dataset=formatted_dataset,  # base_model not with chat_template; dataset with a "text" field or messages + dataset_text_field in config
    train_dataset = train_dataset, # work with instruct_model, default with chat_template;
    args=training_config,
    peft_config=peft_config,  # << enable LoRA: warp peft_config # way-2
)

print("Starting LoRA training…")
lora_trainer.train()

# lora_instruct_model.save_pretrained(lora_instruct_model_name) # way-1-1
# model = PeftModel.from_pretrained(raw_model, lora_instruct_model_name) # way-1-1

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 128012}.


Starting LoRA training…


Step,Training Loss
10,0.761200
20,0.747500
30,0.697200
40,0.688900
50,0.635600
60,0.758000
70,0.726800
80,0.737000
90,0.660600
100,0.641100


TrainOutput(global_step=1000, training_loss=0.6811862144470214, metrics={'train_runtime': 1589.2938, 'train_samples_per_second': 2.517, 'train_steps_per_second': 0.629, 'total_flos': 1.0201008345029837e+17, 'train_loss': 0.6811862144470214})

In [14]:
lora_trainer.save_model() # saved the trained model wrapped with lora parameters
model = AutoModelForCausalLM.from_pretrained(lora_instruct_model_name, device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
test_prompt = "How to learn Agentic AI?"
# Prepare the prompt for instruct model (with chat template)
instruct_messages = [
    {"role": "system", "content": "/think"},
    {"role": "user", "content": test_prompt}
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
instruct_inputs = instruct_tokenizer(instruct_formatted, return_tensors="pt").to(device)

# Generate responses

with torch.no_grad():
    instruct_outputs = model.generate(
        **instruct_inputs,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        pad_token_id=instruct_tokenizer.eos_token_id,
    )
    instruct_response = instruct_tokenizer.decode(
        instruct_outputs[0], skip_special_tokens=False
    )
    # Extract only the assistant's response
    assistant_start = instruct_response.find("<|im_start|>assistant\n") + len(
        "<|im_start|>assistant\n"
    )
    assistant_response = instruct_response[assistant_start:]
    print(assistant_response)

<think>
Okay, the user is asking how to learn Agentic AI. First, I need to define what Agentic AI is. From what I remember, Agentic AI refers to AI systems that have agency, meaning they can make decisions autonomously and take actions based on their goals. They have a sense of self, intentionality, and the ability to reason and adapt.

So, the user probably wants to understand the fundamentals of this field. They might be a student, a researcher, or someone interested in the intersection of AI and ethics. I should start by explaining the key concepts: autonomy, intentionality, self-awareness, and decision-making processes.

Next, I should outline the core principles of Agentic AI. These include the need for self-modeling, goal representation, and interaction with the environment. Self-modeling means the AI can reflect on its own behavior and adjust accordingly. Goal representation is about defining objectives and adjusting them as needed. Interaction with the environment involves sens

### Full Parameter Supervised Fine-tuning

In [16]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_config,
    # train_dataset = formatted_dataset,
    train_dataset=train_dataset,
)

# Start training!
print("\n=== STARTING TRAINING ===")
trainer.train()

# Save the model
trainer.save_model()
print(f"Model saved to {training_config.output_dir}")

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 128012}.



=== STARTING TRAINING ===


Step,Training Loss
10,0.704000
20,0.678600
30,0.624300
40,0.605500
50,0.516500
60,0.687000
70,0.650600
80,0.655100
90,0.572800
100,0.523300


Model saved to ./SmolLM3-Custom-SFT


Acknowledgement: https://huggingface.co/learn/smol-course/unit1/4